# Detecção de Fraudes em Transações com Cartão de Crédito
**Dataset:** Credit Card Fraud Detection — Kaggle  

 



## 1. Instalando e importando as bibliotecas

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print("✅ Tudo importado!")

## 2. Carregando os dados

In [ ]:
df = pd.read_csv('creditcard.csv')
print(f"Linhas e colunas: {df.shape}")
df.head()

## 3. Explorando os dados

In [ ]:
# Quantas fraudes existem?
print(df['Class'].value_counts())
print(f"\nPorcentagem de fraudes: {df['Class'].mean()*100:.4f}%")

In [ ]:
# Gráfico de distribuição das classes
fig, ax = plt.subplots(figsize=(7, 5), facecolor='#0f0f1a')
ax.set_facecolor('#1a1a2e')

contagem = df['Class'].value_counts()
bars = ax.bar(['Legítima', 'Fraude'], contagem.values,
              color=['#00d4ff', '#ff4d6d'], edgecolor='#ffffff22', width=0.5)

for bar, v in zip(bars, contagem.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{v:,}', ha='center', color='white', fontweight='bold')

ax.set_title('Transações Legítimas vs Fraudes', color='white', fontsize=14, pad=15)
ax.set_ylabel('Quantidade', color='#ccccee')
ax.tick_params(colors='#ccccee')
for spine in ax.spines.values():
    spine.set_edgecolor('#444466')

plt.tight_layout()
plt.savefig('distribuicao.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# Comparando o valor médio das transações
medias = df.groupby('Class')['Amount'].mean()
print(f"Valor médio — Legítima: R$ {medias[0]:.2f}")
print(f"Valor médio — Fraude:   R$ {medias[1]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor='#0f0f1a')
for ax, classe, label, cor in zip(axes, [0,1], ['Legítima','Fraude'], ['#00d4ff','#ff4d6d']):
    ax.set_facecolor('#1a1a2e')
    dados = df[df['Class']==classe]['Amount']
    ax.hist(dados, bins=50, color=cor, edgecolor='#ffffff11', alpha=0.85)
    ax.axvline(dados.mean(), color='yellow', linestyle='--', linewidth=1.5, label=f'Média: {dados.mean():.2f}')
    ax.set_title(f'Valores — {label}', color='white', fontsize=12)
    ax.set_xlabel('Valor', color='#ccccee')
    ax.set_ylabel('Frequência', color='#ccccee')
    ax.tick_params(colors='#ccccee')
    ax.legend(fontsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor('#444466')

plt.suptitle('Distribuição dos Valores por Classe', color='white', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('valores.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 4. Preparando os dados para o modelo

In [ ]:
# Normalizando o valor da transação
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])
df['Time']   = scaler.fit_transform(df[['Time']])

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape[0]:,} amostras")
print(f"Teste:  {X_test.shape[0]:,} amostras")

## 5. Treinando o modelo

In [ ]:
modelo = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1,
                                class_weight='balanced')
modelo.fit(X_train, y_train)
print("✅ Modelo treinado!")

## 6. Avaliando o modelo

In [ ]:
y_pred = modelo.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Legítima', 'Fraude']))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5), facecolor='#0f0f1a')
sns.heatmap(cm, annot=True, fmt='d', cmap='magma', ax=ax,
            xticklabels=['Legítima','Fraude'],
            yticklabels=['Legítima','Fraude'],
            linewidths=1, linecolor='#0f0f1a')
ax.set_title('Matriz de Confusão', color='white', fontsize=14, pad=15)
ax.set_xlabel('Previsto', color='#ccccee')
ax.set_ylabel('Real', color='#ccccee')
ax.tick_params(colors='#ccccee')

plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# Features mais importantes para detectar fraude
importancias = pd.Series(modelo.feature_importances_, index=X.columns)
top10 = importancias.nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(9, 6), facecolor='#0f0f1a')
ax.set_facecolor('#1a1a2e')
colors = plt.cm.plasma(np.linspace(0.3, 0.9, len(top10)))
ax.barh(top10.index, top10.values, color=colors, edgecolor='#ffffff11')
ax.set_title('Top 10 Features para Detectar Fraude', color='white', fontsize=13, pad=15)
ax.set_xlabel('Importância', color='#ccccee')
ax.tick_params(colors='#ccccee')
for spine in ax.spines.values():
    spine.set_edgecolor('#444466')

plt.tight_layout()
plt.savefig('features.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 7. Conclusões

- O dataset tem **284.807 transações**, sendo apenas **492 fraudes** (~0.17%) — muito desbalanceado.
- Usamos `class_weight='balanced'` para o modelo lidar com esse desbalanceamento automaticamente.
- O **Random Forest** conseguiu detectar a maioria das fraudes com alta precisão.
- As features **V14, V17 e V12** foram as mais relevantes para identificar transações suspeitas.
